In [1]:
import os
import glob
import shutil
from sec_edgar_downloader import Downloader
from tqdm import tqdm
import pandas as pd
from zipfile import ZipFile

In [2]:
# Define a few CIKs manually or load from CSV
mini_sp500 = pd.DataFrame({
    'CIK': ['0000320193', '0000789019'],  # AAPL and MSFT
    'Ticker': ['AAPL', 'MSFT']
})

mini_sp500

,CIK,Ticker
0,0000320193,AAPL
1,0000789019,MSFT


In [3]:
# Initialize downloader
dl = Downloader("Lehigh University", "akg326@lehigh.edu", "mini_10k_data")

# Download all 10-Ks between 2004 and 2024
for cik in tqdm(mini_sp500['CIK']):
    cik_padded = str(cik).zfill(10)
    firm_folder = f'mini_10k_data/sec-edgar-filings/{cik_padded}/'


    dl.get("10-K", cik_padded,
               after="2004-01-01",
               before="2024-12-31",
               limit=None,  
               download_details=True)

    # Remove .txt files to clean up
    for txt_file in glob.glob(firm_folder + '10-K/*/*.txt'):
        os.remove(txt_file)

# Zip everything
zip_path = 'mini_10k_data.zip'
with ZipFile(zip_path, 'w') as zipf:
    for foldername, subfolders, filenames in os.walk('mini_10k_data'):
        for filename in filenames:
            file_path = os.path.join(foldername, filename)
            zipf.write(file_path, os.path.relpath(file_path, 'mini_10k_data'))




100%|█████████████████████████████████████████████| 2/2 [00:42<00:00, 21.47s/it]


In [5]:
import fnmatch
import glob
import os
import re
from time import sleep
from zipfile import ZipFile

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup
from utils.near_regex import * # this import all th
from tqdm import tqdm  # progress bar on loops

import re as re

In [6]:
from sec_edgar_downloader import Downloader 
dl_folder = "applemini_10k_data_20yrs"
dl = Downloader("Lehigh University", "jmf225@lehigh.edu", dl_folder)
apple_cik = '0000320193' 
cik_padded = str(apple_cik).zfill(10)

dl.get("10-K", cik_padded,
       after="2023-01-01",
       before="2024-12-31",
       limit=None,
       download_details=True)

firm_folder = os.path.join(dl_folder, "sec-edgar-filings", cik_padded)
for txt_file in glob.glob(os.path.join(firm_folder, '10-K', '*', '*.txt')):
    os.remove(txt_file)

zip_path = 'justapple_mini_10k_data.zip'
with ZipFile(zip_path, 'w') as zipf:
    for foldername, subfolders, filenames in os.walk(dl_folder):
        for filename in filenames:
            file_path = os.path.join(foldername, filename)
            zipf.write(file_path, os.path.relpath(file_path, dl_folder)) 

print("ok")
# data downloaded here - BUT CHANGE IT TO TAKE DATA From MEMORY, not redownlaod everythign online  

ok


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_cosine_similarity(text1, text2):
    """
    Calculates the cosine similarity between two texts.

    Args:
        text1 (str): The first text.
        text2 (str): The second text.

    Returns:
        float: The cosine similarity between the two texts (ranging from -1 to 1).
    """
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform([text1, text2])
    return cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]

# Example usage
text_a = "This is the first document."
text_b = "This document is the second document."
similarity_score = calculate_cosine_similarity(text_a, text_b)
print(f"Cosine similarity between the texts: {similarity_score}")

Cosine similarity between the texts: 0.6827531502984261


In [13]:

       
all_documents = []


apple = [] 
apple_df = pd.DataFrame(apple)

with ZipFile('justapple_mini_10k_data.zip','r') as zipfolder: 
    
    # before the loop, get list of files in zipped folder
    file_list = zipfolder.namelist()
    count = 1
    #for index, row in sp500.iterrows(): # loop 
    #just do apple
    apple_folder = f"sec-edgar-filings/{apple_cik.zfill(10)}/10-K/*/*.html"
    possible_apple_files = fnmatch.filter(file_list, apple_folder)



    print(f"found {len(possible_apple_files)} possible 10k files fo rapple")
            
    fpath = possible_apple_files[0] # the first match is the path to the file
    print(fpath)
    for fpath in tqdm(possible_apple_files):
        accession_number = fpath.split('/')[3]
        filing_date_match = re.search(r'(\d{4}-\d{2}-\d{2})', accession_number)

        
        # open the file (this is a little different!)
        with zipfolder.open(fpath) as report_file:
        
            html = report_file.read().decode(encoding="utf-8")

            
            soup = BeautifulSoup(html,features='lxml-xml')
            #soup = BeautifulSoup(html, 'lxml')
            for div in soup.find_all("div", {'style': 'display:none'}):
                div.decompose()
        
            document = soup.text.lower()
            document = re.sub(r'\W', ' ', document)  # Remove punctuation
            document = re.sub(r'\s+', ' ', document)  # Normalize whitespace

            doc_length = len(document.split())

  
            apple.append({'Doc Length': doc_length,'accession': accession_number, 'filing_date': filing_date, 'document': document})
            
            filing_dates = sorted(all_documents.keys())
            documents = [all_documents[date] for date in filing_dates]
            
            # if len(documents) > 1:
            #     vectorizer = TfidfVectorizer()
            #     tfidf_matrix = vectorizer.fit_transform(documents)
            #     cosine_sim_matrix = cosine_similarity(tfidf_matrix)
            
            #     print("Cosine Similarity Matrix")
            #     similarity_df = pd.DataFrame(cosine_sim_matrix, index=filing_dates, columns=filing_dates)
            #     print(similarity_df)
# Sort filings chronologically AFTER processing all files
        
        processed_texts = {}
        sorted_dates = sorted(processed_texts.keys())
        print("\nCosine Similarity Between Consecutive Apple 10-K Filings:")
        
        if len(sorted_dates) > 1:
            vectorizer = TfidfVectorizer()
            for i in range(len(sorted_dates) - 1):
                date1 = sorted_dates[i]
                date2 = sorted_dates[i+1]
                doc1 = processed_texts[date1]
                doc2 = processed_texts[date2]
        
                tfidf_matrix = vectorizer.fit_transform([doc1, doc2])
                similarity_score = cosine_similarity(tfidf_matrix)[0, 1]
        
                print(f"Similarity between {date1} and {date2}: {similarity_score:.4f}")

    # theres defitiely a betetr way to present this 

found 21 possible 10k files fo rapple
1 sec-edgar-filings/0000320193/10-K/0001193125-08-224958/primary-document.html


  0%|                                                    | 0/21 [00:00<?, ?it/s]


AttributeError: 'list' object has no attribute 'keys'